# Customer Health Forensics — Phase 5
## Insight & Decision Intelligence Engine

**Requires:** Phases 1–4 complete (all outputs in Drive)
**LLM:** Mistral-7B via HuggingFace API (or rule fallback)

**What this phase does:**
- ANN feature impact scoring + interaction detection
- RL recommendation optimization
- LLM-powered executive narratives (or rule fallback)
- 8-section intelligence report
- Natural language Q&A over all phase outputs

In [ ]:
!pip install langchain langchain-huggingface jinja2 --quiet
# Optional: enable local LLM
# !pip install transformers accelerate bitsandbytes --quiet
print('Dependencies ready.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/CustomerHealthForensics'
import os, sys
os.makedirs(f'{PROJECT_ROOT}/outputs/insights', exist_ok=True)
sys.path.insert(0, f'{PROJECT_ROOT}/src')
# Set your HuggingFace token for LLM access
# os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
print('Drive mounted.')

In [ ]:
from google.colab import files
print('Upload Phase 5 src files:')
print('  insight_engine.py, reasoning_engine.py, rule_engine.py')
print('  llm_pipeline.py, prompt_templates.py, ann_feature_model.py')
print('  rl_recommender.py, qa_engine.py, insight_runner.py')
print('  Also: feature_engineering.py from Phase 1')
uploaded = files.upload()
for fname, data in uploaded.items():
    dest = f"{PROJECT_ROOT}/src/{fname}"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with open(dest, 'wb') as f: f.write(data)
    print(f'  {fname}')

In [ ]:
# Step 1: Verify all previous phase outputs exist
import os
from pathlib import Path

outputs = Path(f'{PROJECT_ROOT}/outputs')
required = [
    'segmentation/segmentation_results.json',
    'segmentation/global_insights.json',
    'drift/drift_report.json',
    'drift/early_warnings.json',
    'xai/global_explanation.json',
    'xai/watchlist_reasoning.json',
]
for r in required:
    p = outputs / r
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status}  {r}')

In [ ]:
# Step 2 (optional): Train ANN from Phase 1 data
# Uncomment to train (takes ~3 min on T4)
# from ann_feature_model import train_from_phase1_artifacts
# from pathlib import Path
# model = train_from_phase1_artifacts(
#     models_dir = Path(f'{PROJECT_ROOT}/models'),
#     data_dir   = Path(f'{PROJECT_ROOT}/data'),
#     save_path  = Path(f'{PROJECT_ROOT}/models/ann_feature_model.pkl'),
# )
print('ANN step skipped (add training data to enable)')

In [ ]:
# Step 3: Generate full intelligence report (rule fallback mode)
from insight_runner import run_insights
from pathlib import Path

report = run_insights(
    outputs_dir = Path(f'{PROJECT_ROOT}/outputs'),
    models_dir  = Path(f'{PROJECT_ROOT}/models'),
    save_dir    = Path(f'{PROJECT_ROOT}/outputs/insights'),
    llm_mode    = 'rules',  # change to 'api' with HUGGINGFACEHUB_API_TOKEN set
)

In [ ]:
# Step 4: Inspect the report
import json
print('Executive Summary:')
print(report['executive_summary'])
print()
print('Business Impact:')
bi = report['business_impact']
print(f"  Revenue at risk:    ${bi['total_annual_revenue_at_risk']:,.0f}")
print(f"  Potential recovery: ${bi['potential_recovery']:,.0f}")
print(f"  Critical customers: {bi['critical_customers_count']}")
print()
print('Top 3 Recommendations:')
for r in report['recommendations'][:3]:
    print(f"  {r['rank']}. {r['description'][:120]}")

In [ ]:
# Step 5: Q&A system
from qa_engine import QAEngine
from llm_pipeline import LLMPipeline

llm = LLMPipeline(mode='rules')
qa  = QAEngine(outputs_dir=Path(f'{PROJECT_ROOT}/outputs'), llm=llm)

questions = [
    'Why are Pro users churning?',
    'Which segment is at highest risk?',
    'What should we do to reduce churn?',
    'Which features are drifting the most?',
    'Should the model be retrained?',
]
for q in questions:
    result = qa.ask(q)
    print(f'Q: {q}')
    print(f'A: {result["answer"]}')
    print()

In [ ]:
# Step 6: Enable LLM (requires token)
# import os
# os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'hf_YOUR_TOKEN_HERE'
# os.environ['LLM_MODEL_ID'] = 'mistralai/Mistral-7B-Instruct-v0.2'
# 
# from importlib import reload
# import llm_pipeline, insight_engine
# reload(llm_pipeline); reload(insight_engine)
# from insight_runner import run_insights
# report_llm = run_insights(
#     outputs_dir = Path(f'{PROJECT_ROOT}/outputs'),
#     models_dir  = Path(f'{PROJECT_ROOT}/models'),
#     save_dir    = Path(f'{PROJECT_ROOT}/outputs/insights_llm'),
#     llm_mode    = 'api',
# )
print('Uncomment block above and set token to enable LLM mode')

In [ ]:
# Step 7: RL policy
from rl_recommender import RLRecommender

rl = RLRecommender()
print('RL Recommendations for Critical / engagement risk:')
for rec in rl.recommend('Critical', 'engagement', top_k=3):
    print(f"  {rec['rank']}. {rec['action_id']:25s}  {rec['description'][:80]}")

## ✅ Phase 5 Complete

```
outputs/insights/
  intelligence_report.json  ← 8-section report
  executive_summary.txt     ← plain text
  recommendations.json      ← prioritised actions
  qa_log.json               ← 8 standard Q&A answers
  rl_policy.json            ← RL learned actions
```

**Production CLI (no notebook needed):**
```bash
python insight_runner.py                    # full run, rule fallback
python insight_runner.py --llm-mode api     # with HuggingFace API
python insight_runner.py --ask 'Why are Pro users churning?'
python insight_runner.py --train-ann        # train ANN first
```